# Zepto Data Pipeline: Web Scraping to Relational SQLite Store

This notebook demonstrates the end-to-end catalog data engineering pipeline for Zepto:
1. **Scraping**: Extracting product listings from `http://books.toscrape.com/` (100 books across 29 categories).
2. **Cleaning & Transformation**: Parsing raw prices, word-to-number star ratings, and stock status.
3. **Currency Conversion**: Fixed baseline conversion ($1\text{ GBP} = 105.50\text{ INR}$).
4. **Normalized SQLite Storage**: Two-table schema (`categories` and `books`) with Primary Key / Foreign Key constraints.
5. **Analytical SQL Queries**: Covering `WHERE`, `ORDER BY`, `LIMIT`, `DISTINCT`, `IN/BETWEEN`, and `JOIN` with aggregations.
6. **Pandas Equivalence Verification**: Proving that in-memory `pd.merge()` yields the exact same result as the SQL `JOIN`.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup

from scraper import scrape_books_catalog, clean_and_transform_data, FIXED_GBP_TO_INR_RATE
from pipeline import init_database, populate_database, execute_analytical_queries, verify_pandas_merge_equivalence, DB_PATH

print("Modules imported successfully.")

## 1. Web Scraping & Raw Data Extraction
We scrape the first 5 paginated catalog pages from `http://books.toscrape.com/` to retrieve 100 books across diverse categories.

In [ ]:
raw_records = scrape_books_catalog(max_pages=5)
print(f"Total raw records scraped: {len(raw_records)}")
pd.DataFrame(raw_records).head()

## 2. Data Cleaning, Type Casting & Fixed Currency Conversion
- Strip currency symbols -> `price_gbp` (`float64`)
- Star ratings ('One'...'Five') -> `rating` (`int` 1-5)
- Stock availability -> `in_stock` (`bool`)
- Median imputation for numeric errors
- Currency conversion using fixed rate: `1 GBP = 105.50 INR` -> `price_inr`

In [ ]:
clean_df = clean_and_transform_data(raw_records)
print("Cleaned Data Info:")
print(clean_df.info())
clean_df.head(10)

## 3. Normalized Relational Database Loading (`books.db`)
Creating normalized tables `categories` and `books` with Foreign Key constraints.

In [ ]:
conn = init_database(DB_PATH)
populate_database(clean_df, conn)
print("SQLite database initialized and populated at:", DB_PATH)

## 4. Analytical SQL Queries
Executing 6 analytical queries demonstrating filtering, sorting, limiting, distinct values, ranges, and relational joins.

In [ ]:
results = execute_analytical_queries(conn)

for qkey, (desc, sql, df_res) in results.items():
    print("=" * 80)
    print(f"[{qkey}] {desc}")
    print("SQL Query:\n" + sql)
    print(f"Returned {len(df_res)} rows:")
    display(df_res.head(5))

## 5. SQL JOIN vs. In-Memory Pandas Merge Equivalence
Comparing `pd.read_sql` JOIN output against in-memory `pd.merge()` without SQL to prove data pipeline correctness.

In [ ]:
sql_df, pd_df, is_eq = verify_pandas_merge_equivalence(conn)

print("SQL JOIN Output (pd.read_sql):")
display(sql_df)

print("\nPandas Merge Output (pd.merge):")
display(pd_df)

print(f"\nExact Match Verified: {is_eq}")
assert is_eq, "Equivalence check failed!"
print("Assertion Passed: Both approaches produce identical outputs.")

conn.close()